# Minimal cross-store query — Netherlands 2024

Pull **CO2** from the ICOS Obspack zarr and **monthly NEE** from the
FLUXNET zarr for stations inside a Netherlands bounding box, restricted
to 2024.

- **Region**: lat 50.7–53.6, lon 3.3–7.3
- **Time**: 2024-01-01 → 2024-12-31

Each store has a *combined-view group* (built by `combine_to_dim.py`)
where every station shares one `(station, time)` axis pair, so spatial
and temporal selection becomes a single xarray expression — no
per-group loops, no `.zattrs` parsing.

| Store | Combined group | Spatial coords |
|---|---|---|
| `icos-obspack.zarr` | `co2`, `ch4`, `n2o`, `co` | `lat`, `lon`, `intake_height` |
| `icos-fluxnet.zarr` | `_combined/fluxnet_dd` … `_yy` | `lat`, `lon`, `station_elevation` |

In [ ]:
import xarray as xr

LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3
T0, T1 = "2024-01-01", "2024-12-31"

## Obspack — CO2

In [ ]:
ds = xr.open_zarr("icos-obspack.zarr", group="co2", consolidated=False)

co2_nl = (
    ds["co2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time_co2=slice(T0, T1))
)

# Per-station summary — coords travel with the DataArray
for sid in co2_nl.station.values:
    da = co2_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(co2_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(co2_nl.lon.sel(station=sid)):.3f}  "
          f"intake_height={float(co2_nl.intake_height.sel(station=sid)):>4.0f} m  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.2f} ppm")

## FLUXNET — monthly NEE

Selects the VUT/REF NEE variant from each site's `fluxnet_mm` sub-group.

In [ ]:
ds = xr.open_zarr("icos-fluxnet.zarr", group="_combined/fluxnet_mm", consolidated=False)

nee_nl = (
    ds["NEE"]
      .sel(ustar_threshold="VUT", nee_variant="REF")
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time=slice(T0, T1))
)

units = ds["NEE"].attrs.get("units", "")
for sid in nee_nl.station.values:
    da = nee_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(nee_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(nee_nl.lon.sel(station=sid)):.3f}  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.3f} {units}")

## Same query via the data-passport proxy (port 8080)

Reads the stores over HTTP through `run_proxy.py`. Each `.values` access
records chunk fetches; on session close a passport is minted.

Launch the proxy first:

```bash
python run_proxy.py --store-dir . --port 8080
```

In [ ]:
from datapassport_zarr import open_zarr

OBSPACK_URL = "http://localhost:8080/icos-obspack.zarr"
FLUXNET_URL = "http://localhost:8080/icos-fluxnet.zarr"

# Same xarray expressions as above — just over HTTP, with passports.

with open_zarr(OBSPACK_URL, group="co2", verbose=False) as ds:
    co2_nl = (
        ds["co2"]
          .where(
              (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
              (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
              drop=True,
          )
          .sel(time_co2=slice(T0, T1))
    )
    print(f"Obspack CO2 — {len(co2_nl.station)} stations × {len(co2_nl.time_co2)} timestamps")
    print(f"  mean across NL+2024 = {float(co2_nl.mean()):.2f} ppm")

with open_zarr(FLUXNET_URL, group="_combined/fluxnet_mm", verbose=False) as ds:
    nee_nl = (
        ds["NEE"]
          .sel(ustar_threshold="VUT", nee_variant="REF")
          .where(
              (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
              (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
              drop=True,
          )
          .sel(time=slice(T0, T1))
    )
    print(f"\nFluxnet monthly NEE — {len(nee_nl.station)} stations × {len(nee_nl.time)} months")
    print(f"  mean across NL+2024 = {float(nee_nl.mean()):.3f} {ds['NEE'].attrs.get('units','')}")

| Use case | Tool | Why |
|---|---|---|
| `time` slice | `.sel(time=slice(T0, T1))` | `time` is the indexed dim; binary search |
| Lat/lon range | `.where(mask, drop=True)` | `lat`/`lon` are non-index coords on `station` |
| Specific station IDs | `.sel(station=[...])` | `station` is the indexed dim |
| Prefix / regex / any predicate over a coord | build a mask, then `.sel(station=...)` or `.where()` | needs evaluation in Python |




In [ ]:
# AMBITION TO GO TO:

import xarray as xr

ds = xr.open_zarr("icos-obspack.zarr", group="co2")

T0, T1 = "2024-01-01", "2024-12-31"
LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3

# ── 1. Range filter on lat/lon (.where with boolean mask) ─────────────────────
co2_box = (
    ds["co2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time_co2=slice(T0, T1))
)
# Result: 2-D (station, time_co2). Stations outside the box dropped.


# ── 2. Pick specific stations by ID (.sel on the station coord) ───────────────
co2_pick = (
    ds["co2"]
      .sel(station=["CBW27", "CBW67", "CBW127", "CBW207", "JUE50", "JUE80", "JUE120", "LUT0"])
      .sel(time_co2=slice(T0, T1))
)
# Result: 2-D (station, time_co2). Stations explicitly named.


# ── 3. Pick by trigram prefix or any other station-coord predicate ────────────
station_ids = ds.station.values     # 1-D array of station IDs
mask        = [sid.startswith("CBW") or sid.startswith("JUE") for sid in station_ids]
co2_prefix  = (
    ds["co2"]
      .sel(station=station_ids[mask])
      .sel(time_co2=slice(T0, T1))
)
# Result: 2-D (station, time_co2). All Cabauw + Jülich heights.